# ⚠️ STOP: DO NOT RUN THIS ON YOUR MACBOOK!

### How to use this notebook:
1. Go to **[Google Colab](https://colab.research.google.com/)** in your browser.
2. Click **Upload** and upload this `05_finetuning_colab.ipynb` file.
3. In Colab, go to **Runtime > Change runtime type** and select **T4 GPU**.
4. Click the Folder icon on the left sidebar and upload your `train.jsonl` file.
5. Run the cells below!

## 1. Install Unsloth (The fastest fine-tuning library)

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

## 2. Load the Base Model (Llama 3.1 8B)
We use a 4-bit quantized version so it fits on the free Colab GPU.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Maximum context size

# Load Llama 3.1 8B Instruct model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

# Add LoRA adapters (This is what we actually train, it's 10x faster!)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Target modules size
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

## 3. Load Your IP-SAKTI Dataset
Make sure you uploaded `train.jsonl` to the Colab files section!

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

# Set up the chat format
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3",
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }

# Load your dataset
dataset = load_dataset("json", data_files="train.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True,)

print(f"✅ Successfully loaded {len(dataset)} Q&A pairs for training!")

## 4. Train the Model! 🚀
This will take about 10-30 minutes depending on how big your dataset is.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Change this to 150-300 for a real full training run later!
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

## 5. Export for your Mac (.gguf)
This creates the final offline model file. Once finished, download it to your Mac!

In [ ]:
# Export to 4-bit GGUF format (Perfect for Mac M2)
model.save_pretrained_gguf("ip_sakti_model", tokenizer, quantization_method = "q4_k_m")

print("✅ FINISHED! Check the file browser on the left for your ip_sakti_model-unsloth.Q4_K_M.gguf file. Download it to your Mac!")